In [ ]:
# ======================================================================
# LAGGED DUAL-ENCODER multi-window UNet -- Jupyter cell.
#
# Adds a TEMPORAL axis: for each init t0, both encoders see the forecast
# fields from t0, t0-7d, t0-14d, t0-21d, t0-28d, t0-35d.
#
# WHY 6 LAGS AT 7-DAY SPACING
# ---------------------------
# MJO propagates ~5 deg/day with a 30-60 day cycle. Three lags spanning 14
# days cannot distinguish "phase 3 strengthening" from "phase 3 decaying
# from a peak two weeks ago". 35 days covers most of a cycle, which is what
# week5_6 (valid days 29-42) needs -- the MJO relevant to that window may
# currently be a third of the way around the globe.
#
# UNIFORM lags, no padding, no masking: t0-35d exists for every window (the
# archive doesn't care which lead range you're predicting). Only the earliest
# inits of the record lack full history; those rows are DROPPED, not padded.
# The window-id embedding lets the model learn to weight older lags more
# heavily at longer leads without hard-coding it.
#
# LAGS ON BOTH ENCODERS: the regional branch carries total_precipitation
# (your strongest predictor) whose recent trajectory encodes the BSISO
# active/break cycle; the large-scale branch carries the MJO signal itself.
#
# MEMORY (measured, not estimated): the JJAS cache came out 3.5 GB on disk, so
# full year (MONTHS=None, ~3x the inits) lands near 10 GB. That does NOT fit in
# 17 GB alongside the anomalised copies, so the cache is a DIRECTORY of separate
# .npy files and XA/XB are read through a real mmap. Only XAa/XBa -- the arrays
# the DataLoader random-indexes -- stay resident, so peak RSS tracks the
# anomalised size (~10 GB) instead of twice the cache (~20 GB).
#
# NOTE on numerics: the chunked anomalisation is still exact, but _clim_grid is
# no longer bit-identical to the single-matmul version -- it sums partial
# M @ blk products over row chunks, so float32 addition reassociates. Measured
# against the old implementation on synthetic fields with NaNs: NaN pattern
# identical, max abs diff 7.6e-6, and max REL diff 2.8e-6 over cells with
# |clim| > 1. (Whole-array max rel is 1.8e-2, but that is one cell whose
# climatology is 1.9e-5 -- a near-zero denominator, not a real disagreement.)
#
# RUN: args.cmd = "prepare"  then  args.cmd = "train"
# ======================================================================

import gc, json, os, time
from contextlib import contextmanager
import numpy as np

# ---------------- CONFIG ----------------
IMD_TARGET_VAR = "rain"
WINDOWS = [("week2", 8, 14), ("week3_4", 15, 28), ("week5_6", 29, 42)]
LAG_DAYS = [0, 7, 14, 21, 28, 35]      # 0 = the init itself
LAG_TOL_DAYS = 1                        # nearest init within +/- this many days
CLIM_WINDOW_DAYS = 7
COARSE_PAD = 3.0
MONTHS = None                           # None = full year (no month filter)
TEST_YEARS_N = 3
DTYPE = np.float32

BIG_VARS = ["top_net_thermal_radiation", "geopotential_height_200",
            "geopotential_height_500", "geopotential_height_850",
            "geopotential_height_1000"]

CACHE = "../data/cache/unet_cache_lag"                # a DIRECTORY now, not a .npz (see prepare)
OUT_MAPS = "../results/models/unet_lag.nc"


class Args:
    cmd = "train"
    epochs = 60
    batch = 8            # halved: 6x channels per sample
    base = 24
    drop = 0.2
    wd = 1e-3
    lr = 2e-4
    patience = 10
    folds = 5


args = Args()
ARG_DICT = {k: getattr(args, k) for k in dir(args) if not k.startswith("_")}
ARG_DICT["lags"] = LAG_DAYS


@contextmanager
def stage(name):
    print(f"[ ] {name} ...", flush=True)
    t0 = time.perf_counter()
    yield
    print(f"[x] {name}  ({time.perf_counter() - t0:.1f}s)", flush=True)


# ======================================================================
# SHARED
# ======================================================================

def normalize_step(ds, verbose=True):
    v = ds["step"].values
    if np.issubdtype(v.dtype, np.timedelta64):
        return ds
    mx = int(np.nanmax(v))
    u = "D" if mx <= 60 else ("h" if mx <= 24 * 60 else "s")
    if verbose:
        print(f"    step is {v.dtype} (max {mx}) -> units='{u}'")
    td = v.astype(np.int64).astype(f"timedelta64[{u}]").astype("timedelta64[ns]")
    return ds.assign_coords(step=("step", td))


def ensure_valid_time(ds):
    ds = normalize_step(ds)
    expected = ds["time"] + ds["step"]
    if "valid_time" in ds.coords:
        got = ds["valid_time"]
        if got.shape == expected.shape and (got.values == expected.values).all():
            return ds
        print("    WARNING: existing valid_time != time + step -> rebuilding")
        ds = ds.drop_vars("valid_time")
    return ds.assign_coords(valid_time=expected)


def _clim_grid(values, doys, window, rows=None, cell_chunk=20000, row_chunk=512):
    """(n, ...) -> (366, ...) DOY climatology, chunked over cells AND rows.

    `values` may be a np.load(..., mmap_mode="r") memmap, and `rows` selects the
    training subset. Passing rows here instead of slicing at the call site is the
    whole point: values[tr] would materialise ~80% of the array in RAM, which is
    exactly what MONTHS=None cannot afford.

    NOT bit-identical to the single-matmul version any more -- the row loop turns
    one M @ blk into a sum of partial products, so float32 addition reassociates.
    Differences are at the 1e-7 rounding level, not structural.
    """
    shp = values.shape[1:]
    n_all = len(values)
    v2 = values.reshape(n_all, -1)
    n_cells = v2.shape[1]
    if rows is None:
        rows = np.arange(n_all)
    elif np.asarray(rows).dtype == bool:
        rows = np.where(rows)[0]
    rows = np.asarray(rows)
    centers = np.arange(1, 367)
    d = np.abs(doys[rows][None, :].astype(np.int32) - centers[:, None])
    M = (np.minimum(d, 366 - d) <= window).astype(DTYPE)
    out = np.empty((366, n_cells), DTYPE)
    for a in range(0, n_cells, cell_chunk):
        b = min(a + cell_chunk, n_cells)
        counts = np.zeros((366, b - a), DTYPE)
        sums = np.zeros((366, b - a), DTYPE)
        for r0 in range(0, len(rows), row_chunk):
            r1 = min(r0 + row_chunk, len(rows))
            blk = np.asarray(v2[rows[r0:r1], a:b])     # only this slab is resident
            fin = np.isfinite(blk)
            counts += M[:, r0:r1] @ fin.astype(DTYPE)
            sums += M[:, r0:r1] @ np.where(fin, blk, 0).astype(DTYPE)
            del blk, fin
        with np.errstate(invalid="ignore", divide="ignore"):
            out[:, a:b] = np.where(counts > 0, sums / np.maximum(counts, 1), np.nan)
        del counts, sums
    return out.reshape((366,) + shp)


# ======================================================================
# PREPARE -- two boxes, six lags each
# ======================================================================

def _subset_box(ds, imd, pad):
    ds = ensure_valid_time(ds)
    for c in ("lat", "lon"):
        if ds[c].values[0] > ds[c].values[-1]:
            ds = ds.sortby(c)
    if pad is not None:
        la0, la1 = float(imd.lat.min()), float(imd.lat.max())
        lo0, lo1 = float(imd.lon.min()), float(imd.lon.max())
        ds = ds.sel(lat=slice(la0 - pad, la1 + pad), lon=slice(lo0 - pad, lo1 + pad))
    return ds


def _window_stack(sub, feature_vars, leads, lo, hi):
    """Window-mean predictors for ALL inits: (n_init, V, h, w)."""
    from dask.diagnostics import ProgressBar
    sel = np.where((leads >= lo) & (leads <= hi))[0]
    if len(sel) == 0:
        raise ValueError(f"no leads in [{lo},{hi}]")
    with ProgressBar():
        Xw = sub.isel(step=sel).mean(dim="step").compute()
    return np.stack([Xw[v].values for v in feature_vars], axis=1).astype(DTYPE), sel


def _build_lag_index(all_inits, keep_mask, lag_days, tol):
    """For each kept init, find the row index of each lagged init.

    Returns (idx, ok) where idx[i, L] is the row in all_inits closest to
    keep_init[i] - lag_days[L], and ok[i] is True only if EVERY lag was found
    within `tol` days. Rows with ok=False are dropped -- never padded, because
    zero in anomaly space means "exactly climatological", a false claim rather
    than a neutral one.
    """
    day = np.timedelta64(1, "D")
    kept = np.where(keep_mask)[0]
    idx = np.zeros((len(kept), len(lag_days)), np.int64)
    ok = np.ones(len(kept), bool)
    for L, lag in enumerate(lag_days):
        target = all_inits[kept] - lag * day
        pos = np.searchsorted(all_inits, target)
        pos = np.clip(pos, 1, len(all_inits) - 1)
        lo_ = all_inits[pos - 1]
        hi_ = all_inits[np.minimum(pos, len(all_inits) - 1)]
        pick_hi = np.abs(hi_ - target) < np.abs(target - lo_)
        chosen = np.where(pick_hi, np.minimum(pos, len(all_inits) - 1), pos - 1)
        err = np.abs(all_inits[chosen] - target) / day
        ok &= err <= tol
        idx[:, L] = chosen
    return kept, idx, ok


def prepare(ecmwf_ds, big_ds, imd_ds, cache_path):
    import xarray as xr
    from dask.diagnostics import ProgressBar

    imd = imd_ds[IMD_TARGET_VAR]
    imd = imd.assign_coords(time=imd["time"].dt.floor("D"))
    flat_lat, flat_lon = imd["lat"].values, imd["lon"].values

    with stage("Subset both boxes (full year -- lags need pre-season history)"):
        # NOTE: do NOT month-filter here. A JJAS init at t0 needs history back
        # to t0-35d, which for early June reaches into April/May. Filter AFTER
        # building the lag index, on the TARGET init only.
        subA = _subset_box(ecmwf_ds, imd, COARSE_PAD)
        subB = _subset_box(big_ds, imd, pad=None)
        varsA = list(subA.data_vars)
        missing = [v for v in BIG_VARS if v not in subB.data_vars]
        if missing:
            raise KeyError(f"BIG_VARS missing from big_ds: {missing}")
        varsB = BIG_VARS
        clatA, clonA = subA["lat"].values, subA["lon"].values
        clatB, clonB = subB["lat"].values, subB["lon"].values
        leadsA = (subA["step"].values / np.timedelta64(1, "D")).astype(int)
        leadsB = (subB["step"].values / np.timedelta64(1, "D")).astype(int)
        initA, initB = subA["time"].values, subB["time"].values
        print(f"    A: {len(varsA)} vars on {len(clatA)}x{len(clonA)}, "
              f"lat {clatA.min():.1f}..{clatA.max():.1f} lon {clonA.min():.1f}..{clonA.max():.1f}")
        print(f"    B: {len(varsB)} vars on {len(clatB)}x{len(clonB)}, "
              f"lat {clatB.min():.1f}..{clatB.max():.1f} lon {clonB.min():.1f}..{clonB.max():.1f}")

    # --- guards ---
    if initA.shape != initB.shape or not (initA == initB).all():
        raise ValueError("A and B have different init dates -- reindex big_ds first")
    for cla, clo, nm in [(clatA, clonA, "A"), (clatB, clonB, "B")]:
        if not (cla.min() <= flat_lat.min() and cla.max() >= flat_lat.max()
                and clo.min() <= flat_lon.min() and clo.max() >= flat_lon.max()):
            raise ValueError(f"IMD grid not inside box {nm} -- grid_sample would clamp")
    if not (np.diff(initA) > np.timedelta64(0)).all():
        order = np.argsort(initA)
        subA, subB = subA.isel(time=order), subB.isel(time=order)
        initA = initB = initA[order]
        print("    (inits were unsorted -- sorted)")

    with stage("Lag index"):
        month = initA.astype("datetime64[M]").astype(int) % 12 + 1
        keep = np.isin(month, MONTHS) if MONTHS else np.ones(len(initA), bool)
        kept, lag_idx, ok = _build_lag_index(initA, keep, LAG_DAYS, LAG_TOL_DAYS)
        n_drop = int((~ok).sum())
        kept, lag_idx = kept[ok], lag_idx[ok]
        print(f"    {len(kept)} usable inits ({n_drop} dropped for incomplete "
              f"{max(LAG_DAYS)}d history), lags {LAG_DAYS}")
        if len(kept) == 0:
            raise ValueError("no inits have complete history -- widen LAG_TOL_DAYS")

    from numpy.lib.format import open_memmap
    os.makedirs(cache_path, exist_ok=True)
    n_kept, nl, nw = len(kept), len(LAG_DAYS), len(WINDOWS)
    XA_mm = XB_mm = None

    y_l, doy_l, wid_l = [], [], []
    for wi, (wname, lo, hi) in enumerate(WINDOWS):
        with stage(f"Window {wname} (days {lo}-{hi}) x {len(LAG_DAYS)} lags"):
            fullA, sel = _window_stack(subA, varsA, leadsA, lo, hi)
            fullB, _ = _window_stack(subB, varsB, leadsB, lo, hi)
            if XA_mm is None:
                # Allocate on DISK and fill window-by-window. The concatenate
                # version briefly held the per-window list AND the joined array,
                # i.e. 2x the full XA -- unaffordable once MONTHS=None triples it.
                XA_mm = open_memmap(os.path.join(cache_path, "XA.npy"), mode="w+",
                                    dtype=DTYPE, shape=(n_kept * nw, nl) + fullA.shape[1:])
                XB_mm = open_memmap(os.path.join(cache_path, "XB.npy"), mode="w+",
                                    dtype=DTYPE, shape=(n_kept * nw, nl) + fullB.shape[1:])
            # gather lags: (n_kept, n_lag, V, h, w) -- one window resident at a time
            XA_mm[wi * n_kept:(wi + 1) * n_kept] = fullA[lag_idx]
            XB_mm[wi * n_kept:(wi + 1) * n_kept] = fullB[lag_idx]
            del fullA, fullB
            gc.collect()

            vt = subA["valid_time"].isel(time=kept, step=sel).dt.floor("D").values
            with ProgressBar():
                y_all = imd.reindex(time=vt.ravel()).astype(DTYPE).compute().values
            y_l.append(np.nanmean(y_all.reshape(vt.shape + y_all.shape[1:]), axis=1))
            del y_all
            centre = initA[kept] + np.timedelta64((lo + hi) // 2, "D")
            doy_l.append(xr.DataArray(centre, dims="t").dt.dayofyear.values)
            wid_l.append(np.full(len(kept), wi, np.int64))
            print(f"    +{len(kept)} samples")

    XA_mm.flush(); XB_mm.flush()
    y = np.concatenate(y_l); doy = np.concatenate(doy_l); wid = np.concatenate(wid_l)
    year = np.tile(initA[kept].astype("datetime64[Y]").astype(int) + 1970, len(WINDOWS))
    gc.collect()
    print(f"\n    XA {XA_mm.shape} {XA_mm.nbytes/1e9:.2f} GB | "
          f"XB {XB_mm.shape} {XB_mm.nbytes/1e9:.2f} GB | y {y.shape} {y.nbytes/1e9:.2f} GB")

    with stage("Mask + test holdout + cache"):
        mask = np.isfinite(y).all(axis=0)
        uy = np.unique(year)
        test_years = set(uy[-TEST_YEARS_N:])
        is_test = np.isin(year, list(test_years))
        print(f"    strict mask {int(mask.sum())} cells | test {sorted(test_years)}")
        # Separate .npy per big array, NOT one .npz: mmap_mode is silently ignored
        # for npz members, so a savez cache can only ever be read whole into RAM.
        np.save(os.path.join(cache_path, "y.npy"), y)
        np.savez(os.path.join(cache_path, "meta.npz"),
                 doy=doy, wid=wid, year=year,
                 mask=mask, is_test=is_test, clatA=clatA, clonA=clonA,
                 clatB=clatB, clonB=clonB, flat_lat=flat_lat, flat_lon=flat_lon,
                 varsA=np.array(varsA), varsB=np.array(varsB),
                 lag_days=np.array(LAG_DAYS),
                 window_names=np.array([w[0] for w in WINDOWS]))
        tot = sum(os.path.getsize(os.path.join(cache_path, f))
                  for f in os.listdir(cache_path))
        print(f"    {tot/1e9:.2f} GB -> {cache_path}/ (XA.npy, XB.npy, y.npy, meta.npz)")


# ======================================================================
# FOLD-LOCAL PREPROCESSING (chunked; identical results, bounded memory)
# ======================================================================

def anomalise_lagged(X, doy, tr, chunk=256):
    """X: (N, n_lag, V, h, w) -> (N, n_lag*V, h, w), de-climatologised.

    Every (lag, var) channel gets its OWN climatology: after the reshape,
    channel ch = L*V + v, and _clim_grid works per flattened cell, so
    clim[:, ch] is built only from lag-L data. A field 35 days before a June
    init therefore gets a June-minus-35d seasonal mean, not the init's.

    WHY `doy` IS NOT SHIFTED BY lag_days[L] HERE (it looks like a bug, it isn't):
    lag L's field truly sits at doy - lag_days[L], but that offset is CONSTANT
    within a channel, and `doy` is used both to BUILD the bins and to LOOK THEM
    UP. Circular DOY distance is shift-invariant, so grouping by `doy` gives
    exactly the same groups as grouping by doy - lag_days[L] -- only the bin
    label differs, and the label cancels. Verified: applying the shift to both
    build and lookup reproduces this function bit-for-bit (max abs diff 0.0).
    DO NOT "fix" this by shifting only the lookup -- that mismatches label and
    data and moves anomalies by ~2x their own magnitude.

    Chunked + in-place: no full-size clim[doy-1] broadcast, no triple copy.
    """
    N, nl, V, h, w = X.shape
    flat = X.reshape(N, nl * V, h, w)          # X may be a memmap; reshape stays lazy
    clim = _clim_grid(flat, doy, CLIM_WINDOW_DAYS, rows=tr)
    out = np.empty((N, nl * V, h, w), DTYPE)   # not empty_like: X may be a memmap
    for a in range(0, N, chunk):
        b = min(a + chunk, N)
        out[a:b] = flat[a:b] - clim[doy[a:b] - 1]
    del clim

    idx = np.where(tr)[0]
    s1 = np.zeros(nl * V, np.float64); s2 = np.zeros(nl * V, np.float64); cnt = 0
    for a in range(0, len(idx), chunk):
        blk = out[idx[a:a + chunk]]
        s1 += np.nansum(blk, axis=(0, 2, 3))
        s2 += np.nansum(blk.astype(np.float64) ** 2, axis=(0, 2, 3))
        cnt += blk.shape[0] * blk.shape[2] * blk.shape[3]
    m = (s1 / cnt).astype(DTYPE)[None, :, None, None]
    sd = np.sqrt(np.maximum(s2 / cnt - (s1 / cnt) ** 2, 0)).astype(DTYPE)
    sd = np.where(sd < 1e-8, 1.0, sd)[None, :, None, None]
    for a in range(0, N, chunk):
        b = min(a + chunk, N)
        np.subtract(out[a:b], m, out=out[a:b])
        np.divide(out[a:b], sd, out=out[a:b])
        np.nan_to_num(out[a:b], copy=False)
    return out.reshape(N, nl * V, h, w)      # lags folded into channels


def anomalise_target(y, doy, tr, chunk=256):
    clim = _clim_grid(y, doy, CLIM_WINDOW_DAYS, rows=tr)
    out = np.empty(y.shape, DTYPE)
    for a in range(0, len(y), chunk):
        b = min(a + chunk, len(y))
        out[a:b] = y[a:b] - clim[doy[a:b] - 1]
    del clim
    return out


# ======================================================================
# MODEL + TRAIN
# ======================================================================

def build_and_run(args, cache_path, out_maps):
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import DataLoader, Dataset

    dev = ("cuda" if torch.cuda.is_available()
           else "mps" if torch.backends.mps.is_available() else "cpu")

    # XA/XB stay on DISK. They are only ever read sequentially -- anomalise_lagged
    # walks them in row chunks and _clim_grid streams them -- so paging costs little,
    # and it keeps ~10 GB out of RAM at MONTHS=None. The random-access arrays the
    # DataLoader hits are XAa/XBa, which are freshly allocated in RAM downstream.
    # (This is a real mmap because these are .npy; mmap_mode is a no-op on .npz.)
    XA = np.load(os.path.join(cache_path, "XA.npy"), mmap_mode="r")
    XB = np.load(os.path.join(cache_path, "XB.npy"), mmap_mode="r")
    y = np.load(os.path.join(cache_path, "y.npy"))
    z = np.load(os.path.join(cache_path, "meta.npz"))
    doy, wid = np.asarray(z["doy"]), np.asarray(z["wid"])
    year, mask, is_test = np.asarray(z["year"]), np.asarray(z["mask"]), np.asarray(z["is_test"])
    clatA, clonA, clatB, clonB = z["clatA"], z["clonA"], z["clatB"], z["clonB"]
    flat_lat, flat_lon = np.asarray(z["flat_lat"]), np.asarray(z["flat_lon"])
    lag_days = np.asarray(z["lag_days"])
    n_win = len(z["window_names"]); H, W = len(flat_lat), len(flat_lon)
    n_lag = XA.shape[1]
    cA, cB = XA.shape[2] * n_lag, XB.shape[2] * n_lag     # lags folded to channels
    print(f"    device: {dev} | lags {list(lag_days)} | "
          f"encA {cA} ch, encB {cB} ch | {len(XA)} samples")

    def make_samp(clat, clon):
        gy = 2 * (flat_lat - clat[0]) / (clat[-1] - clat[0]) - 1
        gx = 2 * (flat_lon - clon[0]) / (clon[-1] - clon[0]) - 1
        gyy, gxx = np.meshgrid(gy, gx, indexing="ij")
        s = np.stack([gxx, gyy], -1).astype(np.float32)[None]
        assert np.abs(s).max() <= 1.0, "fine grid outside a source box"
        return torch.tensor(s).to(dev)

    sampA, sampB = make_samp(clatA, clonA), make_samp(clatB, clonB)
    lat2 = (flat_lat[:, None] - flat_lat.mean()) / flat_lat.std()
    lon2 = (flat_lon[None, :] - flat_lon.mean()) / flat_lon.std()
    static = torch.tensor(np.stack([mask.astype(DTYPE),
        np.broadcast_to(lat2, (H, W)).astype(DTYPE),
        np.broadcast_to(lon2, (H, W)).astype(DTYPE)])[None]).to(dev)

    def gn(c):
        for g in (8, 4, 2, 1):
            if c % g == 0:
                return nn.GroupNorm(g, c)

    class Block(nn.Module):
        def __init__(s, ci, co, drop=0.0):
            super().__init__()
            L = [nn.Conv2d(ci, co, 3, padding=1), gn(co), nn.SiLU(),
                 nn.Conv2d(co, co, 3, padding=1), gn(co), nn.SiLU()]
            if drop > 0:
                L.append(nn.Dropout2d(drop))
            s.f = nn.Sequential(*L)
        def forward(s, x): return s.f(x)

    class LaggedDualUNet(nn.Module):
        """Lags enter as extra CHANNELS (n_lag x V) on each encoder, so the
        first conv can form differences across lags -- that is the tendency
        signal. A 1x1 'lag mixer' compresses lag-channels before the spatial
        convs, keeping parameter growth sublinear in n_lag."""
        def __init__(s, cA, cB, n_win, base=24, drop=0.2, emb=4):
            super().__init__()
            s.emb = nn.Embedding(n_win, emb)
            s.mixA = nn.Conv2d(cA, base * 2, 1)      # lag/var mixing, cheap
            s.mixB = nn.Conv2d(cB, base * 2, 1)
            s.encA1 = Block(base * 2 + emb, base * 2); s.encA2 = Block(base * 2, base * 2)
            s.encB1 = Block(base * 2, base * 2); s.encB2 = Block(base * 2, base * 2)
            s.inp = Block(base * 4 + 3, base)
            s.d1 = Block(base, base * 2, drop); s.d2 = Block(base * 2, base * 4, drop)
            s.bott = Block(base * 4, base * 4, drop)
            s.u2 = nn.ConvTranspose2d(base * 4, base * 2, 2, stride=2)
            s.du2 = Block(base * 4, base * 2, drop)
            s.u1 = nn.ConvTranspose2d(base * 2, base, 2, stride=2)
            s.du1 = Block(base * 2, base)
            s.head = nn.Conv2d(base, 1, 1); s.pool = nn.MaxPool2d(2)

        def forward(s, xa, xb, wid, sampA, sampB, static):
            b = xa.shape[0]
            a = s.mixA(xa)
            e = s.emb(wid)[:, :, None, None].expand(-1, -1, a.shape[2], a.shape[3])
            ca = s.encA2(s.encA1(torch.cat([a, e], 1)))
            cb = s.encB2(s.encB1(s.mixB(xb)))
            fa = F.grid_sample(ca, sampA.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
            fb = F.grid_sample(cb, sampB.expand(b, -1, -1, -1), mode="bilinear", align_corners=True)
            f = torch.cat([fa, fb, static.expand(b, -1, -1, -1)], 1)
            H0, W0 = f.shape[-2:]
            f = F.pad(f, (0, (-W0) % 4, 0, (-H0) % 4), mode="replicate")
            e0 = s.inp(f); e1 = s.d1(s.pool(e0)); e2 = s.d2(s.pool(e1))
            u = s.du2(torch.cat([s.u2(s.bott(e2)), e1], 1))
            u = s.du1(torch.cat([s.u1(u), e0], 1))
            return s.head(u)[:, :, :H0, :W0].squeeze(1)

    def masked_mse(p, t, m):
        return ((p - t) ** 2 * m).sum() / m.sum().clamp(min=1.0)

    def skill_acc(p, t, fin):
        se_m = np.where(fin, (t - p) ** 2, np.nan); se_c = np.where(fin, t ** 2, np.nan)
        with np.errstate(invalid="ignore"):
            rm = np.sqrt(np.nanmean(se_m, axis=0)); rc = np.sqrt(np.nanmean(se_c, axis=0))
            ok = rc > 1e-6
            skill = np.where(ok, 1 - rm / np.where(ok, rc, 1), np.nan)
            tm = np.nanmean(np.where(fin, t, np.nan), axis=0)
            pm = np.nanmean(np.where(fin, p, np.nan), axis=0)
            num = np.nansum(np.where(fin, (t - tm) * (p - pm), np.nan), axis=0)
            den = np.sqrt(np.nansum(np.where(fin, (t - tm) ** 2, np.nan), axis=0)
                          * np.nansum(np.where(fin, (p - pm) ** 2, np.nan), axis=0))
            acc = np.where(den > 0, num / den, np.nan)
        return skill, acc

    fin_static = mask.astype(DTYPE)     # strict mask (.all(axis=0) in prepare) means
                                         # isfinite(yv) & mask == mask for every sample

    class LazyDS(Dataset):
        """Builds tensors per sample. torch.from_numpy shares memory (unlike
        torch.tensor which copies); yb is built per item so a full-size yt
        array never exists. fin is precomputed once (see fin_static above),
        not recomputed per __getitem__ call."""
        def __init__(s, XAa, XBa, ya, idx):
            s.XAa, s.XBa, s.ya, s.idx = XAa, XBa, ya, idx
        def __len__(s): return len(s.idx)
        def __getitem__(s, i):
            j = s.idx[i]
            return (torch.from_numpy(np.ascontiguousarray(s.XAa[j])),
                    torch.from_numpy(np.ascontiguousarray(s.XBa[j])),
                    int(wid[j]),
                    torch.from_numpy(np.nan_to_num(s.ya[j])),
                    torch.from_numpy(fin_static))

    def train_one(tr_idx, va_idx, XAa, XBa, ya, max_epochs):
        # num_workers=0 on purpose. LazyDS is a closure-local class and macOS's spawn
        # start method cannot pickle it. Hoisting it to module level would fix the
        # pickling, but spawn then COPIES XAa/XBa (~2 GB) into every worker, on both
        # loaders -- a worse problem than the one workers were meant to solve. With the
        # mmap gone and fin precomputed, __getitem__ is a contiguous view plus one
        # 129x135 nan_to_num, so the main thread keeps up between GPU batches.
        tr_dl = DataLoader(LazyDS(XAa, XBa, ya, tr_idx), batch_size=args.batch, shuffle=True)
        va_dl = DataLoader(LazyDS(XAa, XBa, ya, va_idx), batch_size=args.batch, shuffle=False)
        model = LaggedDualUNet(cA, cB, n_win, base=args.base, drop=args.drop).to(dev)
        opt = torch.optim.AdamW(model.parameters(), lr=args.lr, weight_decay=args.wd)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=max_epochs)
        best, best_state, wait = np.inf, None, 0
        for ep in range(max_epochs):
            model.train()
            for xa, xb, wb, yb, mb in tr_dl:
                xa, xb, wb, yb, mb = [t.to(dev) for t in (xa, xb, wb, yb, mb)]
                loss = masked_mse(model(xa, xb, wb, sampA, sampB, static), yb, mb)
                opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            sched.step()
            model.eval(); vl, n = 0.0, 0
            with torch.no_grad():
                for xa, xb, wb, yb, mb in va_dl:
                    xa, xb, wb, yb, mb = [t.to(dev) for t in (xa, xb, wb, yb, mb)]
                    vl += float(masked_mse(model(xa, xb, wb, sampA, sampB, static), yb, mb)) * len(xa)
                    n += len(xa)
            vl /= n
            if vl < best - 1e-6:
                best, wait = vl, 0
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            else:
                wait += 1
            if wait >= args.patience:
                break
        model.load_state_dict(best_state)
        return model, best, ep + 1

    def predict(model, idx, XAa, XBa, ya):
        dl = DataLoader(LazyDS(XAa, XBa, ya, idx), batch_size=args.batch, shuffle=False)
        out = []; model.eval()
        with torch.no_grad():
            for xa, xb, wb, _, _ in dl:
                out.append(model(xa.to(dev), xb.to(dev), wb.to(dev),
                                 sampA, sampB, static).cpu().numpy())
        return np.concatenate(out)

    # ---------- rotating-year CV ----------
    nontest_years = sorted(set(year[~is_test].tolist()))
    blocks = np.array_split(nontest_years, args.folds)
    resume_path = out_maps.replace(".nc", "_folds.json")
    done = {}
    if os.path.exists(resume_path):
        with open(resume_path) as fh:
            done = {int(k): v for k, v in json.load(fh).items()}
        print(f"    resuming folds {sorted(done)}")

    with stage(f"Rotating-year CV: {args.folds} folds over {len(nontest_years)} years"):
        for fi, vy_arr in enumerate(blocks):
            if fi in done:
                r = done[fi]
                print(f"    fold {fi} (cached): skill {r['skill']:+.3f} | ACC {r['acc']:.3f}")
                continue
            vy = set(vy_arr.tolist())
            va_i = np.isin(year, list(vy)) & ~is_test
            tr_i = ~np.isin(year, list(vy)) & ~is_test
            XAa = anomalise_lagged(XA, doy, tr_i)
            XBa = anomalise_lagged(XB, doy, tr_i)
            ya = anomalise_target(y, doy, tr_i)
            model, vloss, eps = train_one(np.where(tr_i)[0], np.where(va_i)[0], XAa, XBa, ya, args.epochs)
            p = predict(model, np.where(va_i)[0], XAa, XBa, ya)
            t = ya[va_i]; fin = np.isfinite(np.asarray(y)[va_i]) & mask[None]
            sk, ac = skill_acc(p, t, fin)
            ms, ma = float(np.nanmean(sk[mask])), float(np.nanmean(ac[mask]))
            done[fi] = {"skill": ms, "acc": ma, "val_years": sorted(vy), "epochs": int(eps)}
            with open(resume_path, "w") as fh:
                json.dump({str(k): v for k, v in done.items()}, fh, indent=2)
            print(f"    fold {fi} val {sorted(vy)}: skill {ms:+.3f} | ACC {ma:.3f} | "
                  f"{eps} ep   [saved]", flush=True)
            del XAa, XBa, ya, model, p, t
            gc.collect()
        ks = [i for i in range(args.folds) if i in done]
        fs = [done[i]["skill"] for i in ks]; fa = [done[i]["acc"] for i in ks]
        print(f"\n    [LAGGED DUAL] CV skill {np.mean(fs):+.4f} +/- {np.std(fs):.4f} | "
              f"CV ACC {np.mean(fa):.4f} +/- {np.std(fa):.4f}")

    # ---------- final model -> test ----------
    with stage("Final model -> test"):
        es_years = set(nontest_years[-2:])
        va_i = np.isin(year, list(es_years)) & ~is_test
        fit_i = (~is_test) & ~va_i
        XAa = anomalise_lagged(XA, doy, fit_i)
        XBa = anomalise_lagged(XB, doy, fit_i)
        ya = anomalise_target(y, doy, fit_i)
        model, _, eps = train_one(np.where(fit_i)[0], np.where(va_i)[0], XAa, XBa, ya, args.epochs)
        te_i = np.where(is_test)[0]
        p = predict(model, te_i, XAa, XBa, ya)
        t = ya[is_test]; fin = np.isfinite(np.asarray(y)[is_test]) & mask[None]
        import xarray as xr
        wid_te = wid[is_test]; dvars = {}
        print(f"    trained {eps} ep")
        for w in range(n_win):
            sm = wid_te == w
            if sm.sum() == 0:
                continue
            sk, ac = skill_acc(p[sm], t[sm], fin[sm])
            wn = str(z["window_names"][w])
            dvars[f"{wn}_skill"] = (("lat", "lon"), sk)
            dvars[f"{wn}_acc"] = (("lat", "lon"), ac)
            print(f"    test {wn:>8}: skill {np.nanmean(sk[mask]):+.4f} | "
                  f"ACC {np.nanmean(ac[mask]):.4f} | {100*np.nanmean(sk[mask]>0):.0f}% cells+")
        out = xr.Dataset(dvars, coords={"lat": flat_lat, "lon": flat_lon})
        out.attrs["model"] = f"lagged dual-encoder, lags={list(lag_days)}d, months={MONTHS}"
        out.to_netcdf(out_maps)
        np.savez_compressed(out_maps.replace(".nc", "_pred.npz"),
                            pred=p, obs=t, wid=wid_te)
        torch.save({"state": model.state_dict(), "args": ARG_DICT},
                   out_maps.replace(".nc", ".pt"))
        print(f"    -> {out_maps} (+ _pred.npz, .pt)")


# ======================================================================
if args.cmd == "prepare":
    prepare(ds_ecmv, ds_big, ds_imd, CACHE)      # noqa: F821
else:
    if not os.path.exists(os.path.join(CACHE, "XA.npy")):
        raise SystemExit(f"run prepare first ({CACHE}/XA.npy missing)")
    build_and_run(args, CACHE, OUT_MAPS)